# **Project: Spotify Streams 2023**

This notebook conducts exploratory data analysis (EDA) on the 'Most Streamed Spotify Songs 2023' dataset sourced from Kaggle. It attempts to detect whether the dataset contains any data quality issues such as corrupted data, NULL values, repeating records, etc. The final output of this notebook will be used to write queries in MySQL and eventually to build a visualization dashboard in PowerBI and a presentation file. *Python is used here strictly for discovery. All actual data cleaning and transformation will be handled in MySQL in the next phase of this ELT pipeline.*

<br>

---

Dataset From: https://www.kaggle.com/datasets/nelgiriyewithana/top-spotify-songs-2023

Github: https://github.com/RinaTheSiamese

Date: April 16, 2026

# **1. Imports**

In [2]:
# import packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

# **2. Load Data**

In [3]:
# Load the dataset, using latin-1 encoding to bypass UnicodeDecodeErrors
file_path = "spotify-2023.csv"
df = pd.read_csv(file_path, encoding='latin-1')

# Verify successful loading
display(df.head())

,track_name,artist(s)_name,artist_count,released_year,released_month,released_day,in_spotify_playlists,in_spotify_charts,streams,in_apple_playlists,...,bpm,key,mode,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%
0,Seven (feat. Latto) (Explicit Ver.),"Latto, Jung Kook",2,2023,7,14,553,147,141381703,43,...,125,B,Major,80,89,83,31,0,8,4
1,LALA,Myke Towers,1,2023,3,23,1474,48,133716286,48,...,92,C#,Major,71,61,74,7,0,10,4
2,vampire,Olivia Rodrigo,1,2023,6,30,1397,113,140003974,94,...,138,F,Major,51,32,53,17,0,31,6
3,Cruel Summer,Taylor Swift,1,2019,8,23,7858,100,800840817,116,...,170,A,Major,55,58,72,11,0,11,15
4,WHERE SHE GOES,Bad Bunny,1,2023,5,18,3133,50,303236322,84,...,144,A,Minor,65,23,80,14,63,11,6


# **3. Data Profiling**

### **3.1 High-Level Overview**

Before anything else, we need to understand the shape of our dataset and what data types Python detected upon loading.

In [4]:
# Display dataset summary
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 953 entries, 0 to 952
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   track_name            953 non-null    object
 1   artist(s)_name        953 non-null    object
 2   artist_count          953 non-null    int64 
 3   released_year         953 non-null    int64 
 4   released_month        953 non-null    int64 
 5   released_day          953 non-null    int64 
 6   in_spotify_playlists  953 non-null    int64 
 7   in_spotify_charts     953 non-null    int64 
 8   streams               953 non-null    object
 9   in_apple_playlists    953 non-null    int64 
 10  in_apple_charts       953 non-null    int64 
 11  in_deezer_playlists   953 non-null    object
 12  in_deezer_charts      953 non-null    int64 
 13  in_shazam_charts      903 non-null    object
 14  bpm                   953 non-null    int64 
 15  key                   858 non-null    ob

**Findings:**
* The 'streams' attribute is registered as an object (string) rather than an int64. It means at least one row contains text or special characters in a column that should be purely numeric.
* The 'in_deezer_playlists' attribute was also registered as an object rather than int64.
* The 'in_shazam_charts' attribute was also registered as an object rather than int64.
* There are 953 total records but 'in_shazam_charts' only returns 903, and 'key' returns 858 keys, signifying that there are empty or NULL entries.

### **3.2 Completeness Check**

Quantify the null values in each column.

---

As mentioned in findings 3.1, 'in_shazam_charts' and 'keys' are expected to contain missing entries.

In [5]:
# Calculate missing values
missing_data = pd.DataFrame({
    'Missing Values': df.isnull().sum()
})

# Display only columns with missing data
display(missing_data[missing_data['Missing Values'] > 0].sort_values(by='Missing Values', ascending=False))

,Missing Values
key,95
in_shazam_charts,50


**Findings:**
* The 'key' column is missing 95 values, and 'in_shazam_charts' is missing 50 values.

### **3.3 Uniqueness Check**

Ensure there are no duplicate records that could artificially inflate our dashboard.

In [29]:
# Check for exact row-level duplicates
exact_duplicates = df.duplicated().sum()
print(f"Total exact duplicate rows: {exact_duplicates}")

Total exact duplicate rows: 0


In [30]:
# Check for logical duplicates (same track name and artist)
logical_duplicates = df[df.duplicated(subset=['track_name', 'artist(s)_name'], keep=False)]

print(f"Tracks appearing more than once: {len(logical_duplicates)}\n")
display(logical_duplicates.sort_values(by='track_name'))

Tracks appearing more than once: 8



,track_name,artist(s)_name,artist_count,released_year,released_month,released_day,in_spotify_playlists,in_spotify_charts,streams,in_apple_playlists,...,bpm,key,mode,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%
372,About Damn Time,Lizzo,1,2022,7,15,2332,2,723894473,0,...,109,A#,Minor,84,72,74,10,0,34,7
764,About Damn Time,Lizzo,1,2022,4,14,9021,0,723894473,242,...,109,A#,Minor,84,72,74,10,0,34,7
178,SNAP,Rosa Linn,1,2022,3,19,3202,18,726307468,148,...,170,NaN,Major,56,53,64,11,0,45,6
873,SNAP,Rosa Linn,1,2022,3,19,1818,0,711366595,3,...,170,NaN,Major,56,52,64,11,0,45,7
345,SPIT IN MY FACE!,ThxSoMch,1,2022,10,31,629,14,303216294,32,...,94,G#,Major,73,65,79,5,2,11,6
482,SPIT IN MY FACE!,ThxSoMch,1,2022,10,31,573,0,301869854,1,...,166,C#,Major,70,57,57,9,20,11,7
512,Take My Breath,The Weeknd,1,2021,8,6,2597,0,130655803,17,...,121,A#,Minor,70,35,77,1,0,26,4
616,Take My Breath,The Weeknd,1,2021,8,6,6392,0,432702334,174,...,121,G#,Major,75,53,74,2,0,11,5


**Findings:**
* Found multiple pairs of logical duplicates (e.g., Lizzo's "About Damn Time", The Weeknd's "Take My Breath"). They have identical track and artist names but different values for other attributes.

### **3.4 Validity Check**

Determine any corrupted data.

In [7]:
# Function to catch common Mojibake from utf-8 read as latin-1
def has_weird_chars(val):
    if not isinstance(val, str): return False
    return bool(re.search(r'[ï¿½ÃÂ]', val))

# Filter rows where either track_name or artist(s)_name has the corrupted symbols
corrupted_records = df[df['track_name'].apply(has_weird_chars) | df['artist(s)_name'].apply(has_weird_chars)]

print(f"Total corrupted records found: {len(corrupted_records)}")
display(corrupted_records[['track_name', 'artist(s)_name']].head())

Total corrupted records found: 105


,track_name,artist(s)_name
21,I Can See You (Taylorï¿½ï¿½ï¿½s Version) (From...,Taylor Swift
26,Calm Down (with Selena Gomez),"Rï¿½ï¿½ma, Selena G"
36,Frï¿½ï¿½gil (feat. Grupo Front,"Yahritza Y Su Esencia, Grupo Frontera"
60,Tï¿½ï¿,"dennis, MC Kevin o Chris"
63,BESO,"Rauw Alejandro, ROSALï¿½"


**Findings:**
* There are 105 records where mojibake characters exists, mostly replacing apostrophes or special alphabet characters. Upon further inspection, this was not caused by a loading error on our part, the dataset already came with broken mojibake characters.

### **3.5 Consistency Check**

Determine any discrepancies in data types.

---

As mentioned in findings 3.1, 'streams', 'in_deezer_charts', and 'in_shazam_charts' were recorded as an object rather than int. We need to find the exact row/s that return strings rather than a numerical value.

In [26]:
# Check 'streams'
# Force conversion to numeric, turning errors into 'NaN'
streams_numeric = pd.to_numeric(df['streams'], errors='coerce')

# Find the rows where the conversion failed
corrupted_streams = df[streams_numeric.isna() & df['streams'].notna()]

print(f"Found {len(corrupted_streams)} row(s) where 'streams' is not a valid number:")
display(corrupted_streams[['track_name', 'artist(s)_name', 'streams']].head())

Found 1 row(s) where 'streams' is not a valid number:


,track_name,artist(s)_name,streams
574,Love Grows (Where My Rosemary Goes),Edison Lighthouse,BPM110KeyAModeMajorDanceability53Valence75Ener...


In [27]:
# Check 'in_deezer_charts'
# Force conversion to numeric, turning errors into 'NaN'
deezer_numeric = pd.to_numeric(df['in_deezer_playlists'], errors='coerce')

# Find the rows where the conversion failed
corrupted_deezer = df[deezer_numeric.isna() & df['in_deezer_playlists'].notna()]

print(f"Found {len(corrupted_deezer)} row(s) where 'in_deezer_playlists' is not a valid number.")
display(corrupted_deezer[['track_name', 'artist(s)_name', 'in_deezer_playlists']].head())

Found 79 row(s) where 'in_deezer_playlists' is not a valid number.


,track_name,artist(s)_name,in_deezer_playlists
48,Starboy,"The Weeknd, Daft Punk","2,445"
54,Another Love,Tom Odell,"3,394"
55,Blinding Lights,The Weeknd,"3,421"
65,Yellow,Chris Molitor,"4,053"
73,Sweater Weather,The Neighbourhood,"1,056"


In [28]:
# Check 'in_shazam_charts'
# Force conversion to numeric, turning errors into 'NaN'
shazam_numeric = pd.to_numeric(df['in_shazam_charts'], errors='coerce')

# Find the rows where the conversion failed
corrupted_shazam = df[shazam_numeric.isna() & df['in_shazam_charts'].notna()]

print(f"Found {len(corrupted_shazam)} row(s) where 'in_shazam_charts' is not a valid number.")
display(corrupted_shazam[['track_name', 'artist(s)_name', 'in_shazam_charts']].head())

Found 7 row(s) where 'in_shazam_charts' is not a valid number.


,track_name,artist(s)_name,in_shazam_charts
12,Flowers,Miley Cyrus,"1,021"
13,Daylight,David Kushner,"1,281"
17,What Was I Made For? [From The Motion Picture ...,Billie Eilish,"1,173"
24,Popular (with Playboi Carti & Madonna) - The I...,"The Weeknd, Madonna, Playboi Carti","1,093"
44,Barbie World (with Aqua) [From Barbie The Album],"Nicki Minaj, Aqua, Ice Spice","1,133"


**Findings:**
* For 'streams', record 574 returned 'BPM110KeyAModeMajorDanceability53Valence75Ener' rather than a numeric value.
* For 'in_deezer_charts' and 'in_shazam_charts', the commas were making it a string value rather than a numeric value.

### **3.6 Sanity Check**

Ensure the dataset's numbers make logical sense.

In [31]:
# Check if month/day constraints are violated
print("Date Logic:")
print(f"Invalid Months (>12): {len(df[(df['released_month'] < 1) | (df['released_month'] > 12)])}")
print(f"Invalid Days (>31): {len(df[(df['released_day'] < 1) | (df['released_day'] > 31)])}")

# Check if days are accurate to the month and year it was released
# Temporary dataframe with standard column names for pd.to_datetime
date_df = df[['released_year', 'released_month', 'released_day']].rename(
    columns={'released_year': 'year', 'released_month': 'month', 'released_day': 'day'}
)

# Attempt to convert to real calendar dates. Impossible dates become 'NaT' (Not a Time)
valid_dates = pd.to_datetime(date_df, errors='coerce')

# Find rows where the date conversion failed
invalid_dates = df[valid_dates.isna()]

print(f"Invalid Calendar Dates (e.g., Feb 30, April 31): {len(invalid_dates)}")
if len(invalid_dates) > 0:
    print("Listing invalid dates:")
    display(invalid_dates[['track_name', 'released_year', 'released_month', 'released_day']])

Date Logic:
Invalid Months (>12): 0
Invalid Days (>31): 0
Invalid Calendar Dates (e.g., Feb 30, April 31): 0


In [32]:
# Check if percentages are valid
percentage_cols = [col for col in df.columns if '%' in col]

print("\nPercentage Bounds:")
for col in percentage_cols:
    out_of_bounds = df[(df[col] < 0) | (df[col] > 100)]

    if len(out_of_bounds) > 0:
        print(f"Invalid {col}: Found {len(out_of_bounds)} rows out of bounds!")
    else:
        print(f"{col}: All values are valid.")


Percentage Bounds:
danceability_%: All values are valid.
valence_%: All values are valid.
energy_%: All values are valid.
acousticness_%: All values are valid.
instrumentalness_%: All values are valid.
liveness_%: All values are valid.
speechiness_%: All values are valid.


**Findings:**
* All days and months make logical sense -not going below or over 31 days and 12 months.
* All percentage values make logical sense -not going below or over 100%.

# ***Any data quality issues identified shall be handled in MySQL.***